# Replace a Step 3 graph colormap

Load a campaign's graph and node indicators from its Step 3 results, then recreate an indicator colormap with plot_graph_colormaps_by_indicator from 3_key_authors_graph.ipynb. The saved image replaces the existing colormap for the selected indicator.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

## Parameters

In [2]:
campaign_name = "Ecuador"

# These defaults match the current Step 3 campaign outputs.
topic_col = "BERTopic_topic_256"
top_percentage = 100
indicator_name = "core_number"
cmap = "YlOrRd"
categorical = False
figsize = (14, 10)
node_size = 40

## Build Step 3 paths

In [3]:
def find_project_root(start_path: str | Path) -> Path:
    """Find the project root containing the src and results directories.

    Args:
        start_path: Directory from which to begin searching.

    Returns:
        Project root directory.

    Raises:
        FileNotFoundError: If no matching project root can be found.
    """
    start_path = Path(start_path).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src").is_dir() and (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a project root with src/ and results/.")


project_root = find_project_root(Path.cwd())
dataset_part = f"{campaign_name}_part_all"
step_3_path = (
    project_root
    / "results"
    / "Labeled_Datasets"
    / campaign_name
    / dataset_part
    / "step_3_key_authors_graph"
    / f"topic_col_{topic_col}_top_percentage_{top_percentage:g}"
)
graph_stem = f"accounts_graph_topic_col_{topic_col}"
graph_path = step_3_path / f"{graph_stem}.graphml"
node_indicators_path = step_3_path / f"{graph_stem}_node_indicators.parquet"
source_notebook_path = project_root / "src" / "3_key_authors_graph.ipynb"
safe_indicator_name = re.sub(
    r"_+",
    "_",
    re.sub(r"[^a-z0-9]+", "_", indicator_name.lower()),
).strip("_")
plot_path = step_3_path / f"graph_{safe_indicator_name}_colormap.png"

paths_df = pd.DataFrame(
    {
        "path": [graph_path, node_indicators_path, plot_path],
        "exists_before_run": [
            graph_path.exists(),
            node_indicators_path.exists(),
            plot_path.exists(),
        ],
    },
    index=["graph", "node indicators", "output plot"],
)
paths_df

,path,exists_before_run
graph,/home/amitner/Backbone-Graph/results/Labeled_D...,True
node indicators,/home/amitner/Backbone-Graph/results/Labeled_D...,True
output plot,/home/amitner/Backbone-Graph/results/Labeled_D...,True


## Load the graph and indicators

In [4]:
missing_inputs = [
    path for path in (graph_path, node_indicators_path) if not path.is_file()
]
if missing_inputs:
    missing_text = "\n".join(f"- {path}" for path in missing_inputs)
    raise FileNotFoundError(f"Missing Step 3 input files:\n{missing_text}")

graph = nx.read_graphml(graph_path)
node_indicators_df = pd.read_parquet(node_indicators_path)
node_indicators_df.index = node_indicators_df.index.astype(str)

if indicator_name not in node_indicators_df.columns:
    available = ", ".join(node_indicators_df.columns.astype(str))
    raise KeyError(
        f"Indicator {indicator_name!r} was not found. Available columns: {available}"
    )

graph_nodes = set(map(str, graph.nodes))
indicator_nodes = set(node_indicators_df.index)
load_summary_df = pd.DataFrame(
    {
        "value": [
            graph.number_of_nodes(),
            graph.number_of_edges(),
            len(node_indicators_df),
            len(graph_nodes & indicator_nodes),
            len(graph_nodes - indicator_nodes),
        ]
    },
    index=[
        "graph nodes",
        "graph edges",
        "indicator rows",
        "matched nodes",
        "graph nodes without indicators",
    ],
)
load_summary_df.style.format({"value": "{:,}"})

,value
graph nodes,"21,923"
graph edges,"610,106"
indicator rows,"21,923"
matched nodes,"21,923"
graph nodes without indicators,0


## Load the plotting function

This reads the function directly from 3_key_authors_graph.ipynb, keeping this repair notebook synchronized with the main implementation.

In [5]:
def write_report(message: str) -> None:
    """Print a plotting progress message.

    Args:
        message: Message to print.
    """
    print(message)


source_notebook = json.loads(source_notebook_path.read_text(encoding="utf-8"))
function_source = next(
    "".join(cell.get("source", []))
    for cell in source_notebook["cells"]
    if cell.get("cell_type") == "code"
    and "def plot_graph_colormaps_by_indicator(" in "".join(
        cell.get("source", [])
    )
)
exec(function_source, globals())
print(f"Loaded plot_graph_colormaps_by_indicator from {source_notebook_path}")

Loaded plot_graph_colormaps_by_indicator from /home/amitner/Backbone-Graph/src/3_key_authors_graph.ipynb


## Replace the colormap plot

In [ ]:
indicator_label = indicator_name.replace("_", " ").title()
plot_graph_colormaps_by_indicator(
    graph=graph,
    node_indicator_dict=node_indicators_df[indicator_name].to_dict(),
    output_path=plot_path,
    indicator_name=indicator_label,
    cmap=cmap,
    categorical=categorical,
    figsize=figsize,
    node_size=node_size,
)
print(f"Replaced colormap plot: {plot_path}")

Starting plot_graph_colormaps_by_indicator: nodes=21,923, indicator=Core Number, output_path=/home/amitner/Backbone-Graph/results/Labeled_Datasets/Ecuador/Ecuador_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_100/graph_core_number_colormap.png
